In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('/kaggle/input/final-dataset/finalDF (1).csv')
df.head()

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure
0,plan risk management,studies,project risk,define the plan risk management concept. risk ...,NaN,"described in section 13.1.3.1,described in sec...",plan identify ambiguity,NaN
1,plan risk management,studies,project risk,define the plan risk management concept. the r...,NaN,"described in section 11.2.3.2,described in sec...",plan identify ambiguity,NaN
2,plan risk management inputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN
3,plan risk management tools and techniques,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN
4,plan risk management outputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN


In [5]:
!pip install networkx plotly rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.4/562.4 kB 8.9 MB/s eta 0:00:00a 0:00:01


In [6]:
from rdflib import Graph, URIRef, Literal, Namespace
import pandas as pd
import networkx as nx
import plotly.graph_objs as go


# Initialize an RDF graph
rdf_graph = Graph()
EX = Namespace("")

# Add data to RDF graph
for _, row in df.iterrows():
    subject = URIRef(EX[row['Concepts'].replace(" ", "_")])
    predicate = URIRef(EX[row['Type_relation'].replace(" ", "_")])
    obj = URIRef(EX[row['Concept_of_type_relation'].replace(" ", "_")])
    rdf_graph.add((subject, predicate, obj))

# Save the graph to a file (optional)


In [7]:
G = nx.DiGraph()
for s, p, o in rdf_graph:
    G.add_edge(str(s), str(o), label=str(p).split('/')[-1])

# 3D Positioning for nodes
pos_3d = nx.spring_layout(G, dim=3, seed=42)

def add_line_breaks(text, words_per_line=5):
    words = text.split()
    lines = [" ".join(words[i:i+words_per_line]) for i in range(0, len(words), words_per_line)]
    return "<br>".join(lines)

# Extract node positions
x_nodes = [pos_3d[node][0] for node in G.nodes()]
y_nodes = [pos_3d[node][1] for node in G.nodes()]
z_nodes = [pos_3d[node][2] for node in G.nodes()]
node_labels = [{"Concepts :"+ str(df.loc[i, "Concepts"] ) 
              +': <br> Definition: '+add_line_breaks(str(df.loc[i, "Definition"] ), words_per_line=10)
              +'<br> Synonym: '+ str(df.loc[i, "Synonym"] )
              + "<br> Process Name: "+ str(df.loc[i, "Process_name"]) }for i in range(len(df))]
edges_labels = list(df.Type_relation)

# Extract edge positions
edge_x = []
edge_y = []
edge_z = []
for edge in G.edges():
    edge_x += [pos_3d[edge[0]][0], pos_3d[edge[1]][0], None]
    edge_y += [pos_3d[edge[0]][1], pos_3d[edge[1]][1], None]
    edge_z += [pos_3d[edge[0]][2], pos_3d[edge[1]][2], None]

# Create Plotly traces
edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    line=dict(width=6, color='black'),
    hovertext=edges_labels,
    mode='lines',
    hoverinfo="text"
)

node_trace = go.Scatter3d(
    x=x_nodes, y=y_nodes, z=z_nodes,
    mode='markers+text',
    marker=dict(size=10, color='red', opacity=1),
    hovertext=node_labels,
    textposition="middle center",
    hoverinfo="text",
)

# Layout and plot
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    scene=dict(
        xaxis=dict(showbackground=False),
        yaxis=dict(showbackground=False),
        zaxis=dict(showbackground=False)
    ),
    title='3D Visualization of RDF Graph',
      showlegend=True,
    hoverlabel=dict(
        bgcolor="white",    # Background color for hover label
        font_size=12,       # Font size
        font_family="Arial", # Font family
        bordercolor="black"  # Border color around the hover label
    )
)

fig.show()

In [8]:
!pip install node2vec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 38.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.3 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
libpysal 4.9.2 requires packaging>=22, but you have packaging 21.3 which is incompatible.
libpysal 4.9.2 requires shapely>=2.0.1, but you have shapely 1.8.5.post1 which is incompatible.
tsfresh 0.20.3 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.


In [9]:
import networkx as nx
from node2vec import Node2Vec


# Initialize Node2Vec
node2vec = Node2Vec(G, dimensions=64, walk_length=30, num_walks=200, p=1, q=1, workers=4)

# Fit and transform the model
model = node2vec.fit(window=10, min_count=1, batch_words=4)

# Save model for future use
model.save("node2vec.model")

Generating walks (CPU: 4): 100%|██████████| 50/50 [00:00<00:00, 84.51it/s]


In [10]:
nodes_embedding = []
for i in range(len(df)):
    nodes_embedding.append(model.wv[list(G.nodes()).index(df.loc[i, "Concepts"].replace(" ", "_", -1))])

In [11]:
Concept_of_type_relation_embedding = []
for i in range(len(df)):
    Concept_of_type_relation_embedding.append(model.wv[list(G.nodes()).index(df.loc[i, "Concept_of_type_relation"].replace(" ", "_", -1))])

In [12]:
df['Nodes Embedding'] = nodes_embedding
df['Concept of Type Relation Embedding'] = Concept_of_type_relation_embedding

df.head()

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure,Nodes Embedding,Concept of Type Relation Embedding
0,plan risk management,studies,project risk,define the plan risk management concept. risk ...,NaN,"described in section 13.1.3.1,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930..."
1,plan risk management,studies,project risk,define the plan risk management concept. the r...,NaN,"described in section 11.2.3.2,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930..."
2,plan risk management inputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.37153205, -0.46170852, 0.3147171, 0.6941603...","[0.73298645, -0.15584536, 0.2066423, -0.348930..."
3,plan risk management tools and techniques,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.25836927, -0.21494018, -0.38267234, 0.28611...","[0.73298645, -0.15584536, 0.2066423, -0.348930..."
4,plan risk management outputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.30888477, -0.20805754, 0.18658443, 0.484829...","[0.73298645, -0.15584536, 0.2066423, -0.348930..."


In [13]:
from gensim.models import KeyedVectors

def get_edge_embedding(model, vec1, vec2, method='average'):
    
    if method == 'average':
        return (vec1 + vec2) / 2
    elif method == 'hadamard':
        return vec1 * vec2
    elif method == 'weighted_l1':
        return abs(vec1 - vec2)
    elif method == 'weighted_l2':
        return (vec1 - vec2) ** 2

In [14]:
edge_embedding = []
for i in range(len(df)):
    node1 = model.wv[list(G.nodes()).index(df.loc[i, "Concepts"].replace(" ", "_", -1))]
    node2 = model.wv[list(G.nodes()).index(df.loc[i, "Concept_of_type_relation"].replace(" ", "_", -1))]
    edge_embedding.append(get_edge_embedding(model, node1, node2, method='weighted_l2'))

In [15]:
df['Edge Embedding'] = edge_embedding
df.head()

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure,Nodes Embedding,Concept of Type Relation Embedding,Edge Embedding
0,plan risk management,studies,project risk,define the plan risk management concept. risk ...,NaN,"described in section 13.1.3.1,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.35558876, 0.05308024, 0.25296813, 0.3217625..."
1,plan risk management,studies,project risk,define the plan risk management concept. the r...,NaN,"described in section 11.2.3.2,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.35558876, 0.05308024, 0.25296813, 0.3217625..."
2,plan risk management inputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.37153205, -0.46170852, 0.3147171, 0.6941603...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.13064928, 0.09355226, 0.011680165, 1.088038..."
3,plan risk management tools and techniques,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.25836927, -0.21494018, -0.38267234, 0.28611...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.22526146, 0.0034921973, 0.34729174, 0.40328..."
4,plan risk management outputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.30888477, -0.20805754, 0.18658443, 0.484829...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.17986223, 0.0027261116, 0.00040231823, 0.69..."


In [16]:


from rdflib import Graph, URIRef, Literal, Namespace
import pandas as pd
import networkx as nx
import plotly.graph_objs as go


# Initialize an RDF graph
rdf_graph = Graph()
EX = Namespace("")

# Add data to RDF graph
for _, row in df.iterrows():
    subject = Literal(row['Nodes Embedding'])
    predicate = Literal(row['Edge Embedding'])
    obj = Literal(row['Concept of Type Relation Embedding'])
    rdf_graph.add((subject, predicate, obj))




In [17]:
df = df[df['Concepts'].notna()]
df = df[df['Concept_of_type_relation'].notna()]


In [18]:
G = nx.from_pandas_edgelist(df, 'Concepts', 'Concept_of_type_relation', edge_attr='Type_relation')

for _, row in df.iterrows():
    if row['Definition']:
        G.nodes[row['Concepts']]['x'] = (list(G.nodes()).index(row['Concepts']), row['Definition'])
    else:
        G.nodes[row['Concepts']]['x'] = (list(G.nodes()).index(row['Concepts']), "other")

    if row['Process_name']:
        G.nodes[row['Concept_of_type_relation']]['x'] = (list(G.nodes()).index(row['Concepts']), row['Definition'])
    else:
        G.nodes[row['Concept_of_type_relation']]['x'] = (list(G.nodes()).index(row['Concepts']), "other")
        

In [19]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.3 MB/s eta 0:00:00a 0:00:01


In [20]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 3.7 MB/s eta 0:00:00 0:00:01


In [21]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Charger le modèle pré-entraîné
embb_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
embb_model.eval()

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.73k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning:

`clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884



1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [22]:
defention_embedding = []
for i in range(len(df)):
    defention_embedding.append(embb_model.encode(str(df.loc[i, "Definition"]), convert_to_tensor=False) )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
Process_name_embedding = []
for i in range(len(df)):
    Process_name_embedding.append(embb_model.encode(str(df.loc[i, "Process_name"]), convert_to_tensor=False) )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
df['Defention Embedding'] = defention_embedding
df['Process_name_embedding'] = Process_name_embedding

df.head()

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure,Nodes Embedding,Concept of Type Relation Embedding,Edge Embedding,Defention Embedding,Process_name_embedding
0,plan risk management,studies,project risk,define the plan risk management concept. risk ...,NaN,"described in section 13.1.3.1,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.35558876, 0.05308024, 0.25296813, 0.3217625...","[-0.20070007, 0.029315392, -0.16247118, -0.259...","[-0.47751307, 0.30824012, -0.37095937, -0.3750..."
1,plan risk management,studies,project risk,define the plan risk management concept. the r...,NaN,"described in section 11.2.3.2,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.35558876, 0.05308024, 0.25296813, 0.3217625...","[-0.3061132, 0.17728311, -0.018800333, -0.1801...","[-0.47751307, 0.30824012, -0.37095937, -0.3750..."
2,plan risk management inputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.37153205, -0.46170852, 0.3147171, 0.6941603...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.13064928, 0.09355226, 0.011680165, 1.088038...","[-0.46067682, 0.525866, -0.02412639, -0.413401...","[-0.47751307, 0.30824012, -0.37095937, -0.3750..."
3,plan risk management tools and techniques,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.25836927, -0.21494018, -0.38267234, 0.28611...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.22526146, 0.0034921973, 0.34729174, 0.40328...","[-0.46067682, 0.525866, -0.02412639, -0.413401...","[-0.47751307, 0.30824012, -0.37095937, -0.3750..."
4,plan risk management outputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.30888477, -0.20805754, 0.18658443, 0.484829...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.17986223, 0.0027261116, 0.00040231823, 0.69...","[-0.46067682, 0.525866, -0.02412639, -0.413401...","[-0.47751307, 0.30824012, -0.37095937, -0.3750..."


In [26]:
start_node = []
for _, row in df.iterrows():
    start_node.append(list(G.nodes()).index(row['Concepts']))
    
    
end_node = []
for _, row in df.iterrows():
    end_node.append(list(G.nodes()).index(row['Concept_of_type_relation']))
    
    
df['start_node'] = start_node
df['end_node'] = end_node

df.head()

,Concepts,Type_relation,Concept_of_type_relation,Definition,Synonym,Reference,Process_name,Figure,Nodes Embedding,Concept of Type Relation Embedding,Edge Embedding,Defention Embedding,Process_name_embedding,start_node,end_node
0,plan risk management,studies,project risk,define the plan risk management concept. risk ...,NaN,"described in section 13.1.3.1,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.35558876, 0.05308024, 0.25296813, 0.3217625...","[-0.20070007, 0.029315392, -0.16247118, -0.259...","[-0.47751307, 0.30824012, -0.37095937, -0.3750...",0,1
1,plan risk management,studies,project risk,define the plan risk management concept. the r...,NaN,"described in section 11.2.3.2,described in sec...",plan identify ambiguity,NaN,"[0.13667382, -0.38623685, 0.70960164, 0.218310...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.35558876, 0.05308024, 0.25296813, 0.3217625...","[-0.3061132, 0.17728311, -0.018800333, -0.1801...","[-0.47751307, 0.30824012, -0.37095937, -0.3750...",0,1
2,plan risk management inputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.37153205, -0.46170852, 0.3147171, 0.6941603...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.13064928, 0.09355226, 0.011680165, 1.088038...","[-0.46067682, 0.525866, -0.02412639, -0.413401...","[-0.47751307, 0.30824012, -0.37095937, -0.3750...",2,1
3,plan risk management tools and techniques,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.25836927, -0.21494018, -0.38267234, 0.28611...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.22526146, 0.0034921973, 0.34729174, 0.40328...","[-0.46067682, 0.525866, -0.02412639, -0.413401...","[-0.47751307, 0.30824012, -0.37095937, -0.3750...",3,1
4,plan risk management outputs,studies,project risk,NaN,NaN,NaN,plan identify ambiguity,NaN,"[0.30888477, -0.20805754, 0.18658443, 0.484829...","[0.73298645, -0.15584536, 0.2066423, -0.348930...","[0.17986223, 0.0027261116, 0.00040231823, 0.69...","[-0.46067682, 0.525866, -0.02412639, -0.413401...","[-0.47751307, 0.30824012, -0.37095937, -0.3750...",4,1


In [27]:
import torch
from torch_geometric.data import Data

edge_index = torch.tensor([df['start_node'].values, df['end_node'].values] , dtype=torch.long)

# Edge features
edge_features = torch.tensor([list(i) for i in list(df['Edge Embedding'].values)], dtype=torch.float)

# If you also have node features in the DataFrame, define them
node_features = torch.tensor([list(i) for i in list(df['Defention Embedding'].values)], dtype=torch.float)

target = torch.tensor(df['end_node'].values, dtype=torch.long)


# Create the PyTorch Geometric data object
data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features, y=target )

/tmp/ipykernel_69/3642633147.py:4: UserWarning:

Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)



In [39]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(384, 384)
        self.conv2 = GCNConv(384, 153)
        

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        print(x)
       
        return x


from torch_geometric.loader import DataLoader

# Create a DataLoader if you have multiple graphs
# loader = DataLoader([data], batch_size=1, shuffle=True)

model = GNN(in_channels=1, hidden_channels=4, out_channels=2)  # Adjust channels as needed
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()  # For classification tasks

# Training loop
model.train()
for epoch in range(200):  # Number of epochs
    optimizer.zero_grad()
    out = model(data)  # Forward pass
    loss = criterion(out, data.y)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update weights

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

tensor([[-6.8132e-02, -4.5486e-04,  1.7693e-01,  ..., -1.6148e-01,
         -1.8252e-01, -1.2755e-01],
        [-9.2444e-01,  6.3060e-01,  5.5805e-01,  ..., -1.3116e+00,
         -1.9172e+00, -8.6413e-01],
        [-3.0044e-01,  5.2068e-03, -1.2328e-01,  ..., -4.7552e-01,
          9.7467e-02, -1.1442e-01],
        ...,
        [-1.0984e-01,  6.0825e-02,  2.2461e-01,  ...,  1.4478e-01,
         -2.4925e-01,  3.2765e-02],
        [-6.5467e-02, -1.3367e-01,  9.1436e-02,  ..., -1.0011e-01,
         -2.7955e-01, -4.2282e-02],
        [-7.0415e-02,  1.1407e-01, -1.4438e-01,  ..., -2.3747e-01,
         -3.5910e-01, -1.8497e-01]], grad_fn=<AddBackward0>)
Epoch 1, Loss: 4.974666118621826
tensor([[ -0.9000,   4.3700,  -0.7824,  ...,  -1.1026,  -1.7384,  -1.3270],
        [-11.6822,  42.5906,  -7.5023,  ..., -10.9621, -13.5998, -10.7987],
        [ -2.4018,   6.7077,  -1.0718,  ...,  -2.0850,  -1.0985,  -1.1167],
        ...,
        [ -0.8702,   3.1332,  -0.3621,  ...,  -0.7115,  -1.3451,  -0.8